# Scratch: try out preprocessing and models

A place to experiment.

When something works, move the clean version into `src/credit_risk/models.py`
and check it with `pytest tests/test_models.py`.

**Workflow**

1. Sections 1 and 2 show what the preprocessor does to each type of feature, and fit one model by hand.
2. Section 3 sets up cross-validation once: shared stratified folds, metrics and two helpers.
3. Section 4 runs a baseline: several model families at default settings on the same data and folds.
4. Section 5 compares them: validation scores, the train/validation gap, then a chart.

In [1]:
# Reload src/ code on every run, so edits to models.py show up without restarting.
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from credit_risk.data import TARGET, load_split
from credit_risk.preprocess import build_preprocessor

train = load_split("train")
X_train, y_train = train.drop(columns=TARGET), train[TARGET]
X_train.shape, y_train.mean()

((24000, 24), np.float64(0.22120833333333334))

## 1. What does the preprocessor do?

In [3]:
pre = build_preprocessor()
Xt = pre.fit_transform(X_train)
print(X_train.shape, "->", Xt.shape)
pre.get_feature_names_out()[:15]

(24000, 24) -> (24000, 28)


array(['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
       'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2',
       'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'SEX_2'],
      dtype=object)

### What happens to each type of feature

24 input columns become 28 output columns. The rules live in `src/credit_risk/preprocess.py`.

| Group | Columns | What the preprocessor does | Output columns | Why |
|---|---|---|---|---|
| Numeric amounts | `LIMIT_BAL`, `AGE`, `BILL_AMT1-6`, `PAY_AMT1-6` (14) | `StandardScaler`: subtract the training mean, divide by the training std | 14, same names | Puts NT$ amounts and years on one scale, so logistic regression's coefficients and regularization treat them evenly. Tree models split on thresholds, so scaling does not change them. |
| Categorical codes | `SEX`, `EDUCATION`, `MARRIAGE` (3) | `OneHotEncoder`: one 0/1 column per level. `drop="if_binary"` keeps one column for 2-level `SEX`. `handle_unknown="ignore"` turns an unseen level into all zeros | 8: `SEX_2`, `EDUCATION_1-4`, `MARRIAGE_1-3` | The codes are labels. Without one-hot, a linear model would read `EDUCATION = 4` as "more" than `EDUCATION = 1`. |
| Ordinal repayment status | `PAY_1-6` (6) | Passed through unchanged | 6, same names | -2 to 8 runs roughly from "paid early / no use" to "8 months late", so the order carries meaning. Left unscaled. |
| Other | `ID` | Dropped (`remainder="drop"`) | 0 | A row label, not information about the client. |

### What `Xt` is, and why CV does not reuse it

`pre.fit_transform(X_train)` does two things:

1. **fit**: learns settings from all 24,000 training rows (the scaler's mean and std per column, the encoder's list of levels).
2. **transform**: applies those settings to the same rows.

So `Xt` is the full training set, transformed with statistics that have seen every training row. That is the right setup for the **final model**: once a model is chosen, the pipeline (preprocessor + model) is fit once on all of train, then scored once on the test split.

It is the wrong input for **cross-validation**. If folds were cut from `Xt`, each validation fold would already be scaled with a mean and std that include its own rows. That is a small leak of validation information into training. With a plain scaler on 24,000 rows the effect is likely small, but the habit matters once preprocessing learns more from the data (imputation, target encoding, feature selection).

Section 3 avoids this by putting `build_preprocessor()` inside the pipeline, so `cross_validate` refits it on each set of 4 training folds and only applies it to the held-out fold. `Xt` here is for looking at the output and for the quick fit in section 2.

## 2. Fit a model by hand

One fit on all of train, scored on the same rows. Useful to see what `fit` and
`predict_proba` return, but the score is optimistic. Section 3 does it properly.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Before training, weights are placeholders; each iteration of coefficients calculates a loss,
# and directs each weight to lower it; nudge in that direction;
# the goal is to reach convergence where the loss no longer improves
m = LogisticRegression(max_iter=1000).fit(Xt, y_train)
proba = m.predict_proba(Xt)[:, 1]
print(proba[:5])
print("train AUC:", roc_auc_score(y_train, proba))  # on training data, so optimistic

[0.50744388 0.14212182 0.2423766  0.13209962 0.23678692]
train AUC: 0.7250465202142095


## 3. Cross-validation setup

Run this section once. It defines:

- `cv`: one set of 5 stratified folds that every model is scored on.
- `scoring`: the metrics from the M2 plan.
- `run_cv(name, model)`: cross-validates one model and stores its scores in `results`.
- `compare()`: a coloured table of every model in `results`, side by side.

### Stratifying is not balancing

These two ideas are easy to mix up. They act at different stages.

| | Stratifying | Balancing |
|---|---|---|
| What it changes | How rows are **divided** into folds (or train/test) | What the **model learns from**: row weights, or resampled rows |
| Default rate the model sees | Stays at the real ~22% | Pushed toward 50/50, in effect |
| Goal | Every fold looks like the whole dataset, so fold scores are comparable and `val_std` reflects the model, not a lucky split | Make the model pay more attention to the minority class (defaulters) |
| Where it is used here | `StratifiedKFold` below, and `stratify=` in the train/test split in `data.py` | Not used in the baseline (see section 4) |

**Why stratify.** With 22% defaults, an unstratified 5-fold split can by chance give one fold 20% and another 24%. Recall and precision would then shift partly because the fold changed, not because the model did. Stratifying fixes each fold at about 22%, so differences between folds and between models are easier to read. It does not change the class imbalance itself.

**Balancing** is a modelling choice. It is left out of the baseline for now; section 4 has the reasons.

In [5]:
import warnings

import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline

from credit_risk.config import default_config_path, load_config

# The dummy never predicts "default", so precision is 0/0 for it. Expected; hide the warning.
warnings.filterwarnings("ignore", message="Precision is ill-defined")

cfg = load_config(default_config_path())

# One set of folds shared by every model, so score differences come from the
# model, not from which rows landed in which fold. Stratified: each fold keeps
# the ~22% default rate.
cv = StratifiedKFold(n_splits=cfg.cv_folds, shuffle=True, random_state=cfg.seed)

# Metrics from the M2 plan. Accuracy left out: predicting "no default" for
# everyone already scores 0.78. average_precision is PR-AUC.
scoring = ["roc_auc", "average_precision", "recall", "precision", "f1"]

results = {}  # model name -> its summary table; filled by run_cv


def run_cv(name, model):
    """Cross-validate `model` behind the preprocessor on the shared folds;
    store and show its scores."""
    # Preprocessor inside the pipeline: it is refit on each training fold,
    # so the held-out fold never influences the scaling.
    pipe = make_pipeline(build_preprocessor(), model)
    scores = pd.DataFrame(
        cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)
    )  # one row per fold

    # One row per metric. val = the held-out fold; train = the folds it was fit on.
    summary = pd.DataFrame(
        {
            "val_mean": [scores[f"test_{m}"].mean() for m in scoring],
            "val_std": [scores[f"test_{m}"].std() for m in scoring],  # spread across folds
            "train_mean": [scores[f"train_{m}"].mean() for m in scoring],
        },
        index=scoring,
    )
    summary["gap"] = summary["train_mean"] - summary["val_mean"]  # large = overfitting

    results[name] = summary
    print(f"{name}: {scores['fit_time'].mean():.2f}s to fit per fold")
    return summary.round(3)


def _shade(score):
    """Blue cell background, darker for higher scores (scores are on a 0-1 scale)."""
    return f"background-color: rgba(49, 130, 189, {0.75 * score:.2f})"


def compare():
    """Mean validation score of every model run so far. Rows = metrics, columns = models."""
    table = pd.DataFrame({name: s["val_mean"] for name, s in results.items()})
    return (
        table.style.format("{:.3f}")
        .map(_shade)  # lab4-style colouring, without needing matplotlib
        .highlight_max(axis=1, props="font-weight: bold")  # best model per metric
    )

## 4. Models

Add or remove entries in `models`, then run the cell. Each one is
cross-validated on the same folds and stored in `results` under its name.
Use a distinct name per setting, for example `"logreg_C0.1"`.

### What this experiment is for

Every model gets the **same data, the same preprocessing, the same 5 folds, and default settings**. Holding everything else fixed means score differences come from the model family. That gives a baseline to measure later work against:

- **Feature engineering**: does a new feature raise validation scores by more than `val_std` (the fold-to-fold noise)? If not, it may not be a real improvement.
- **Hyperparameter search**: does tuning beat the default? The default score is the bar to clear.
- **Model choice**: which families are worth tuning at all. There is little point tuning a model that sits well below the others at default settings.

### The models and what their default settings mean

| Name | Model | Key settings (default unless noted) | What they mean |
|---|---|---|---|
| `dummy` | `DummyClassifier(strategy="prior")` | predicts the training default rate (~0.22) for every client | Ignores the features. ROC-AUC is 0.5 and recall is 0. The floor: a real model has to beat this. |
| `logreg` | `LogisticRegression` | `penalty="l2"`, `C=1.0`, `solver="lbfgs"`, `max_iter=1000` (set by us; default is 100) | A weighted sum of features passed through a sigmoid. `C` is the inverse of regularization strength: smaller `C` shrinks coefficients more. `max_iter` is raised so the solver reaches convergence. Linear, so it cannot model interactions or thresholds unless features are built for it. |
| `forest` | `RandomForestClassifier` | `n_estimators=100`, `max_depth=None`, `min_samples_leaf=1`, `max_features="sqrt"`, `bootstrap=True` | 100 decision trees, each fit on a bootstrap sample of rows and considering a random subset of features (√28 ≈ 5) at each split. The prediction is the average of the trees. With no depth limit and leaves of 1 row, each tree can memorize its training rows. |
| `lgbm` | `LGBMClassifier` | `n_estimators=100`, `learning_rate=0.1`, `num_leaves=31`, `max_depth=-1` (no limit), `min_child_samples=20` | Gradient boosting: trees are built one after another, each one correcting the errors of the ones before. `learning_rate` scales each tree's contribution. `num_leaves` caps tree size. `verbose=-1` only silences log output. |
| `xgboost` | `XGBClassifier` | `n_estimators=100`, `learning_rate=0.3`, `max_depth=6` | Also gradient boosting, with a larger step size than LightGBM's default and depth-limited trees. `random_state=cfg.seed` set by us, to match the others. |

### Class balancing: left out for now

The data is imbalanced (~22% defaulters), but the baseline uses no balancing (`class_weight`, resampling).

In [6]:
from lightgbm import LGBMClassifier
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "dummy": DummyClassifier(strategy="prior"),  # floor: ignores the features
    "logreg": LogisticRegression(max_iter=1000),  # first real model
    "forest": RandomForestClassifier(random_state=cfg.seed),
    "lgbm": LGBMClassifier(verbose=-1, random_state=cfg.seed),
    "xgboost": XGBClassifier(random_state=cfg.seed),
}

results = {}  # start fresh, so removed models drop out of the comparison
for name, model in models.items():
    run_cv(name, model)

dummy: 0.01s to fit per fold
logreg: 0.03s to fit per fold
forest: 3.01s to fit per fold
lgbm: 0.63s to fit per fold
xgboost: 0.27s to fit per fold


## 5. Compare

### 5.1 Table

Mean validation score per metric (rows) and model (columns). Darker = higher;
bold = best on that row.

This table only shows `val_mean`. `run_cv` also stores `val_std`, `train_mean`
and `gap` for each model in `results[name]` (for example `results["forest"]`),
but they are not displayed here. The next cell puts `gap` side by side for all models.

In [7]:
compare()

,dummy,logreg,forest,lgbm,xgboost
roc_auc,0.500,0.722,0.759,0.776,0.760
average_precision,0.221,0.503,0.532,0.555,0.523
recall,0.000,0.236,0.361,0.361,0.367
precision,0.000,0.720,0.646,0.667,0.629
f1,0.000,0.356,0.463,0.468,0.464


### 5.2 Train/validation gap

`gap` = train score minus validation score, for the same metric.

- **train** is the model scored on the 4 folds it was fit on; **validation** is the held-out fold.
- Near 0: the model does about as well on new rows as on the rows it learned from.
- Large: the model fits its training rows much better than new ones (overfitting).

`val_std` is the fold-to-fold spread. A gap much larger than `val_std` is not noise.

How to read it:

- A large gap does not by itself make a model worse. Rank on validation score; the forest still beats logreg on validation despite its gap.
- The forest's train score of 1.000 is expected: unlimited depth and 1-row leaves let each tree memorize its training rows.
- Logreg's near-zero gap means it is not overfitting. Its limit is that it is too simple (underfitting), so more regularization would not help; better features might.
- For the boosted models and the forest, the gap suggests room for hyperparameter search that restricts them (`max_depth`, `min_samples_leaf` / `min_child_samples`, lower `learning_rate` with more trees). The aim is a smaller gap without losing validation score.

In [8]:
# Train minus validation, per metric (rows) and model (columns). Darker = bigger gap.
gaps = pd.DataFrame({name: s["gap"] for name, s in results.items()})
gaps.style.format("{:.3f}").map(_shade)

,dummy,logreg,forest,lgbm,xgboost
roc_auc,0.000,0.003,0.241,0.117,0.198
average_precision,0.000,0.002,0.468,0.182,0.366
recall,0.000,0.002,0.638,0.076,0.245
precision,0.000,0.001,0.354,0.121,0.282
f1,0.000,0.002,0.536,0.094,0.269


### 5.3 Chart

Grouped bars: metrics on the x-axis, one colour per model. Hover a bar for its
numbers; the black line is ±1 std across the 5 folds.

In [9]:
import altair as alt

# Long format: one row per (model, metric)
df = pd.concat(results, names=["model", "metric"]).reset_index()
df["low"], df["high"] = df["val_mean"] - df["val_std"], df["val_mean"] + df["val_std"]

base = alt.Chart(df).encode(
    x=alt.X("metric:N", sort=scoring, title=None, axis=alt.Axis(labelAngle=-45)),
    xOffset="model:N",  # places models side by side within each metric
    color=alt.Color("model:N", title="Model"),
)
bars = base.mark_bar().encode(
    y=alt.Y(
        "val_mean:Q", scale=alt.Scale(domain=[0, 1]), title="Validation score (mean of 5 folds)"
    ),
    tooltip=[
        "model:N",
        "metric:N",  # hover shows these
        alt.Tooltip("val_mean:Q", format=".3f"),
        alt.Tooltip("val_std:Q", format=".3f"),
    ],
)
spread = base.mark_rule(color="black").encode(y="low:Q", y2="high:Q")  # ±1 std across folds

(bars + spread).properties(width=450, height=280).configure_axis(
    labelFontSize=13, titleFontSize=14
).configure_legend(labelFontSize=13, titleFontSize=14)

alt.LayerChart(...)

### 5.4 Takeaways: how the model families differ

**Which are trees?** `forest`, `xgboost` and `lgbm` are all built from decision trees. `logreg` is a linear model, and `dummy` ignores the features.

| Family | How it works | Analogy | Main risk |
|---|---|---|---|
| Decision tree (building block) | A flowchart of yes/no splits, e.g. `PAY_0 >= 2`, then `LIMIT_BAL < 50k`. A leaf predicts the default rate of its training rows. | A loan officer's checklist | One deep tree memorizes its rows |
| Random forest (bagging) | Many deep trees, **trained independently** on random rows and features, then averaged. Averaging cancels much of each tree's individual error (lowers variance). | A crowd guessing jellybeans in a jar, then averaging | Large train/val gap unless trees are restricted |
| Gradient boosting (XGBoost, LightGBM) | Small trees **trained in sequence**, each fitted to the errors left by the trees before it and scaled by `learning_rate` (lowers bias). | A golfer's putts, each correcting the last miss | Overfits with too many rounds or too large a step |
| Logistic regression | A weighted sum of features through a sigmoid. | A points-based scorecard | Can't capture thresholds or interactions without engineered features |

**XGBoost vs LightGBM:** same algorithm with different engines. XGBoost grows trees level by level (`max_depth=6`), LightGBM grows the leaf that helps most (`num_leaves=31`). Their defaults also differ: `learning_rate` 0.3 vs 0.1.

**What the baseline showed (ROC-AUC, val / train):**

- `lgbm` 0.776 / 0.893: best validation score and the smallest gap of the tree models.
- `xgboost` 0.760 / 0.959 and `forest` 0.759 / 1.000: similar validation scores but larger gaps. XGBoost's larger default step size is a likely cause of its gap, so this is a tuning question rather than evidence that one library is better.
- `logreg` 0.722 / 0.725: stable but underfitting. The tree models' advantage suggests threshold effects (e.g. months late on `PAY_*`) matter here.

**Next:** tune the boosted models first (lower `learning_rate` with more trees, `min_child_samples`, `num_leaves` / `max_depth`) and check whether the gap shrinks without losing validation score.

### 5.5 How the trees learn: purity, stopping, and combining

**One tree: split to make purer groups.** At each node the tree tries every feature and threshold and keeps the split that makes the two child groups most "pure" with respect to the actual `y`. For sklearn trees (and so the forest), purity is measured with **Gini impurity**:

$$\text{Gini} = 1 - \sum_k p_k^2 \;=\; 2p(1-p) \text{ for two classes, where } p = \text{share of defaulters}$$

| Group | p (default rate) | Gini |
|---|---|---|
| All defaulters or all non-defaulters | 1.0 or 0.0 | 0 (pure) |
| This dataset overall | 0.22 | 0.34 |
| 50/50 mix | 0.5 | 0.5 (least pure) |

A split is scored by the drop from the parent's Gini to the **row-weighted average** of the children's Gini. For example, 1,000 rows at 22% default (Gini 0.343) split on `PAY_0 >= 2`:

- yes: 100 rows at 70% default, Gini 0.42
- no: 900 rows at 16.7% default, Gini 0.278
- weighted: 0.1 × 0.42 + 0.9 × 0.278 = 0.292, so the impurity decrease is 0.343 - 0.292 = **0.051**

The split with the largest decrease wins, and the process repeats inside each child. A leaf predicts the share of defaulters among its training rows.

**When a tree stops splitting.** A node becomes a leaf when a limit is hit or no split lowers impurity:

| Model | Limits at default settings | Effect |
|---|---|---|
| `forest` (each tree) | `max_depth=None`, `min_samples_split=2`, `min_samples_leaf=1`, `min_impurity_decrease=0` | Keeps splitting until leaves are pure or can't be split, so trees memorize their rows (train AUC 1.000) |
| `lgbm` | `num_leaves=31`, `min_child_samples=20`, `min_split_gain=0` | At most 31 leaves, each with 20+ rows |
| `xgboost` | `max_depth=6`, `min_child_weight=1`, `gamma=0` | At most 6 levels deep |

Raising these limits (smaller depth, bigger minimum leaf size, a minimum gain) is how trees are regularized.

**Random forest: an average of all trees, not the last one.** The 100 trees are independent: each is trained on the actual `y` using its own bootstrap sample of rows and random feature subsets, and no tree sees another's errors. `predict_proba` **averages the 100 trees' probabilities**. With 1-row leaves each tree outputs mostly 0 or 1, so if 37 of 100 trees put a client in a "default" leaf, the forest predicts about 0.37. Order doesn't matter; there is no "last tree".

**Boosting: a sum of all trees, each fitted to the errors so far.** Here the loss is explicit: **log loss** between the actual `y` and the predicted probability `p`.

1. Start every client at the base rate: p = 0.22, score = log-odds = -1.27.
2. Compute the residual `y - p` per client (+0.78 for a defaulter, -0.22 for a non-defaulter). This is the negative gradient of log loss.
3. Fit a small regression tree to the residuals, not to `y`. Its splits group clients with similar residuals (XGBoost and LightGBM score splits by the gain in the loss rather than by Gini).
4. Add `learning_rate × leaf value` to each client's score, recompute `p = sigmoid(score)`, and repeat.

The final prediction is `sigmoid(base + lr·tree1 + lr·tree2 + ... + lr·tree100)`. Each tree is one small correction, so later trees focus on the clients the model still gets wrong.

| | Random forest | Gradient boosting |
|---|---|---|
| Each tree predicts | the actual `y` | the current residuals `y - p` |
| Trees depend on each other | No | Yes |
| Combined by | **averaging** probabilities | **summing** scaled scores, then sigmoid |
| Tree size | deep (strong but noisy) | shallow (weak correction) |

## 6. Try your `get_model`

Raises until you implement it in `src/credit_risk/models.py`. Save the file and
re-run this cell. Once it works, `models` in section 4 can use `get_model("logreg", seed=cfg.seed)`
in place of the hand-written models.

In [2]:
from credit_risk.models import get_model

for name in ["dummy", "forest", "logreg", "lgbm", "xgboost"]:
    print(name, get_model(name, seed=0))

dummy DummyClassifier(random_state=0)
forest RandomForestClassifier(random_state=0)
logreg LogisticRegression(max_iter=1000, random_state=0)
lgbm LGBMClassifier(random_state=0, verbose=-1)
xgboost XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)
